# Clase 058 — Optuna: HPO bayesiano dedicado

TPE sampler + MedianPruner sobre GradientBoostingClassifier. Comparamos vs GridSearchCV con presupuesto similar.

Instalar: `pip install optuna`.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import StratifiedKFold, GridSearchCV, cross_val_score
from sklearn.metrics import roc_auc_score
import time

try:
    import optuna
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    OPTUNA_OK = True
except ImportError:
    print('optuna no instalado: `pip install optuna`')
    OPTUNA_OK = False

np.random.seed(42)

## 1. Dataset sintético

In [ ]:
X, y = make_classification(n_samples=2000, n_features=20, n_informative=10,
                            n_redundant=5, weights=[0.7, 0.3], random_state=42)
print('X', X.shape, 'pos rate', y.mean().round(3))
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

## 2. Objective + TPE sampler + MedianPruner

In [ ]:
def objective(trial):
    params = {
        'n_estimators':      trial.suggest_int('n_estimators', 50, 300),
        'max_depth':         trial.suggest_int('max_depth', 2, 8),
        'learning_rate':     trial.suggest_float('learning_rate', 1e-3, 0.3, log=True),
        'subsample':         trial.suggest_float('subsample', 0.5, 1.0),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 20),
    }
    aucs = []
    for step, (tr, te) in enumerate(cv.split(X, y)):
        model = GradientBoostingClassifier(random_state=42, **params)
        model.fit(X[tr], y[tr])
        auc = roc_auc_score(y[te], model.predict_proba(X[te])[:, 1])
        aucs.append(auc)
        trial.report(np.mean(aucs), step)
        if trial.should_prune():
            raise optuna.TrialPruned()
    return float(np.mean(aucs))

In [ ]:
if OPTUNA_OK:
    sampler = optuna.samplers.TPESampler(seed=42)
    pruner = optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=1)
    study = optuna.create_study(direction='maximize', sampler=sampler, pruner=pruner)
    t0 = time.perf_counter()
    study.optimize(objective, n_trials=50, show_progress_bar=False)
    t_optuna = time.perf_counter() - t0
    print(f'Optuna best AUC: {study.best_value:.4f}')
    print(f'Optuna best params: {study.best_params}')
    print(f'completed {len([t for t in study.trials if t.state.name == "COMPLETE"])}/50, '
          f'pruned {len([t for t in study.trials if t.state.name == "PRUNED"])}, time {t_optuna:.1f}s')

## 3. GridSearchCV con presupuesto similar (~48 combos)

In [ ]:
param_grid = {
    'n_estimators':  [50, 150, 300],
    'max_depth':     [3, 5],
    'learning_rate': [0.01, 0.1],
    'subsample':     [0.7, 1.0],
    'min_samples_split': [2, 10],
}
# 3 × 2 × 2 × 2 × 2 = 48 combos

t0 = time.perf_counter()
gs = GridSearchCV(GradientBoostingClassifier(random_state=42),
                   param_grid, cv=cv, scoring='roc_auc', n_jobs=1)
gs.fit(X, y)
t_grid = time.perf_counter() - t0
print(f'Grid best AUC: {gs.best_score_:.4f}')
print(f'Grid best params: {gs.best_params_}')
print(f'48 combos, time {t_grid:.1f}s')

## 4. Comparativa

In [ ]:
if OPTUNA_OK:
    summary = pd.DataFrame({
        'estrategia': ['GridSearchCV (48)', 'Optuna TPE (50)'],
        'AUC':  [gs.best_score_, study.best_value],
        'time_s': [t_grid, t_optuna],
    }).round(4)
    print(summary.to_string(index=False))

## 5. Param importances (fANOVA, matplotlib)

In [ ]:
if OPTUNA_OK:
    importances = optuna.importance.get_param_importances(study)
    names = list(importances.keys())
    vals = list(importances.values())
    fig, ax = plt.subplots(figsize=(7, 3))
    ax.barh(names[::-1], vals[::-1], color='#37a')
    ax.set_xlabel('importancia fANOVA')
    ax.set_title('Importancia de hiperparámetros')
    plt.tight_layout()
    plt.show()

## 6. Historia de optimización

In [ ]:
if OPTUNA_OK:
    values = [t.value for t in study.trials if t.value is not None]
    best_so_far = np.maximum.accumulate(values)
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.scatter(range(len(values)), values, alpha=0.5, label='trial AUC')
    ax.plot(range(len(values)), best_so_far, color='red', label='best so far')
    ax.set_xlabel('trial')
    ax.set_ylabel('AUC')
    ax.set_title('Optuna optimization history')
    ax.legend()
    plt.tight_layout()
    plt.show()

## Ejercicios

1. Cambiá a `CmaEsSampler` (puramente continuo). ¿Converge más rápido?
2. Implementá multi-objective (`directions=['maximize', 'minimize']`) optimizando AUC y tiempo de inferencia.
3. Persistí el study con `storage='sqlite:///opt.db'` y verificá `load_if_exists=True`.

## Conclusiones

- TPE explora el espacio de forma inteligente — alcanza mejor o igual AUC que Grid con menos trials efectivos.
- `MedianPruner` mata trials malos temprano — ahorra cómputo sin perder buenos candidatos.
- `plot_param_importances` orienta dónde poner esfuerzo — típicamente `learning_rate` y `max_depth` dominan en GBM.